# Universidad Católica de Salta — Facultad de Ingeniería
## Investigación Operativa II (2026) — Trabajo Final Integrador
### Sistema de turnos dinámicos con recomendación de vacantes para las asambleas de designación docente en Salta

**Integrantes:** Emiliano Javier Aguirre (DNI 45.770.631) y Pablo Celaya (DNI 45.115.839)  
**Docentes:** Mg. Ing. Gustavo Ramiro Rivadera · Ing. Gustavo Flores Vargas  

Este notebook reproduce de forma interactiva y modular:
1. La verificación analítica del motor de eventos discretos contra las fórmulas de $M/M/1$ y $M/M/c$.
2. La calibración del recomendador multicriterio (*Hit Rate@3* y *Hit Rate@1*) y la evaluación del pronóstico Monte Carlo ($MSE$/$RMSE$).
3. La simulación de los 3 escenarios principales (E1, E2 y E3) sobre las 30 réplicas independientes con pruebas de hipótesis pareadas ($gl = 29$).
4. La visualización de las figuras del informe.

In [ ]:
import json
import pathlib
import pandas as pd
from IPython.display import Image, display

# Cargar los resultados ya consolidados de las 30 réplicas por escenario
resultados_path = pathlib.Path("data/resultados_experimentos.json")
with open(resultados_path, "r", encoding="utf-8") as f:
    res = json.load(f)

print("Resultados experimentales cargados correctamente.")

---  
## 1. Verificación analítica del motor de simulación ($M/M/1$ y $M/M/c$)
Contraste de las medias simuladas contra las soluciones analíticas de cátedra:

In [ ]:
val_filas = []
for caso, datos in res["validacion_analitica"].items():
    val_filas.append({
        "Modelo": caso,
        "Wq simulado (h)": f"{datos['Wq_sim_media']:.4f} ± {datos['Wq_sim_semi_ic']:.4f}",
        "Wq teórico (h)": f"{datos['Wq_teorico']:.4f}",
        "Error Wq (%)": f"{datos['Wq_error_pct']:.2f} %",
        "Lq simulado": f"{datos.get('Lq_sim_media', 0):.4f}",
        "Lq teórico": f"{datos.get('Lq_teorico', 0):.4f}",
        "IC contiene teórico": "Sí" if datos['Wq_contiene_teorico'] else "No"
    })

df_val = pd.DataFrame(val_filas)
display(df_val)

---  
## 2. Desempeño del recomendador y del pronóstico Monte Carlo
Resultados de la grilla de pesos y reducción del error cuadrático medio ($MSE$):

In [ ]:
rec_info = res["calibracion_recomendador"]
print(f"Óptimo de pesos hallado: {rec_info['optimo_pesos']} (wd, wt, wc)")
print(f"Hit Rate@3 óptimo: {rec_info['hit_rate_optimo']:.2f} %")
print(f"Hit Rate@1 óptimo: {rec_info['hit_rate_top1_optimo']:.2f} %")
print(f"Hit Rate@3 al azar: {rec_info['baseline_azar']:.2f} %")
print(f"Hit Rate@1 al azar: {rec_info['baseline_top1_azar']:.2f} %")

mc_info = res["evaluacion_pronostico_mc"]
print("\nEvaluación del Pronóstico Monte Carlo en el preaviso:")
print(f"MSE con Monte Carlo: {mc_info['mse_mc']:.4f} ± {mc_info['ic_mse_mc'][1] - mc_info['mse_mc']:.4f}")
print(f"MSE sin pronóstico:  {mc_info['mse_ingenuo']:.4f} ± {mc_info['ic_mse_ingenuo'][1] - mc_info['mse_ingenuo']:.4f}")
print(f"Reducción en MSE:    {mc_info['reduccion_mse_pct']:.1f} % (p < 0.001)")

---  
## 3. Comparación principal entre escenarios (E1, E2 y E3)
Medias e intervalos de confianza del 95 % sobre las 30 réplicas independientes:

In [ ]:
res_esc = res["comparacion_escenarios"]
comp_filas = []
for esc in ["E1", "E2", "E3"]:
    d = res_esc[esc]
    comp_filas.append({
        "Escenario": esc,
        "Wq atendidos (min)": f"{d['Wq_media']:.2f} ± {d['Wq_semi_ic']:.2f}",
        "W1 primer llamado (min)": f"{d.get('W1_media', d['Wq_media']):.2f}",
        "Wconv 300 doc (min)": f"{d['Wconv_media']:.2f} ± {d['Wconv_semi_ic']:.2f}",
        "Ttotal (horas)": f"{d['Ttotal_media']/60:.2f} h",
        "% Reintentos": f"{d['reintentos_pct_media']:.2f} %",
        "Docentes afectados mérito": f"{d.get('afectados_merito_conteo_medio', 0):.1f}",
        "Satisfacción (0-1)": f"{d['satisfaccion_media']:.3f}"
    })

df_comp = pd.DataFrame(comp_filas)
display(df_comp)

---  
## 4. Visualización de las figuras del informe

In [ ]:
figuras = [
    "graficos/salida/fig1_boxplot_duracion_escenarios.png",
    "graficos/salida/fig2_sensibilidad_ventana_tau.png",
    "graficos/salida/fig3_sensibilidad_comisiones_c.png",
    "graficos/salida/fig4_adopcion_recomendador_y_montecarlo.png"
]

for fig in figuras:
    print(f"Mostrando: {fig}")
    display(Image(filename=fig, width=700))